# Module 7.2 — HyDE (Hypothetical Document Embeddings)

**Problem**: A short user query is semantically distant from long documents in the corpus.

**HyDE solution**:
1. Use LLM to generate a *hypothetical* answer document
2. Embed the hypothetical document (not the query)
3. Retrieve documents similar to the hypothetical answer

Bridges the **query–document semantic gap**.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

docs = [
    Document(page_content="BERT uses bidirectional context to understand language, trained with masked language modelling."),
    Document(page_content="GPT-4 is a decoder-only autoregressive model that predicts the next token."),
    Document(page_content="T5 treats every NLP task as a text-to-text problem."),
    Document(page_content="Llama 3 is Meta's open-weights model family available in 8B and 70B sizes."),
    Document(page_content="RoBERTa improves on BERT by training longer with more data and removing the next sentence prediction task."),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="hyde_demo")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

hyde_prompt = ChatPromptTemplate.from_template("""
Write a short, factual paragraph (3-4 sentences) that would answer this question.
Do NOT say you don't know — write a plausible, detailed response.

Question: {question}

Hypothetical answer:
""")

def hyde_retrieve(question: str, k: int = 3) -> tuple[str, list[Document]]:
    # Generate hypothetical document
    hyp_doc = (hyde_prompt | llm | StrOutputParser()).invoke({"question": question})
    # Embed hypothetical doc, not the query
    hyp_vec = embeddings.embed_query(hyp_doc)
    results = vs.similarity_search_by_vector(hyp_vec, k=k)
    return hyp_doc, results

query = "How does BERT understand language context?"

# ── Standard retrieval ────────────────────────────────────────────────────────
std_results = vs.similarity_search(query, k=3)
print("Standard retrieval:")
for d in std_results:
    print(f"  • {d.page_content[:80]}")

# ── HyDE retrieval ────────────────────────────────────────────────────────────
hyp_doc, hyde_results = hyde_retrieve(query)
print(f"\nHypothetical document generated:\n  {hyp_doc[:200].strip()}")
print("\nHyDE retrieval:")
for d in hyde_results:
    print(f"  • {d.page_content[:80]}")
